<a href="https://colab.research.google.com/github/RasanaPJ/Deep-Learning/blob/master/Custom_MCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Custom MCP Tools Integration with LangChain MCP Adapter

This notebook demonstrates how to integrate custom MCP tools with LangChain using `langchain-mcp-adapters`. The MCP server is defined in `stock_price_server.py` and connected via `MultiServerMCPClient` using stdio transport.

## What This Notebook Shows

**Model Context Protocol (MCP)** is an open standard that lets any LLM agent (Claude Desktop, Cursor, LangChain, custom ADK agents, ...) discover and call tools that live in a separate process. Instead of hard-coding tools into one agent, you publish them once on an MCP server and every compatible agent can use them.

Two pieces work together here:

| File | Role |
|------|------|
| `stock_mcp.py` | **MCP server** — exposes a `get_stock_price` tool over SSE on port 8100. Built with `FastMCP`. |
| This notebook  | **MCP client** — connects to the server, lists its tools, and lets a LangChain ReAct agent call them via `langchain-mcp-adapters`. |

**Run order:** start the server first in a separate terminal (`python stock_mcp.py`), *then* execute the notebook cells. The notebook will fail to connect if the server is not already running on `http://127.0.0.1:8100/sse`.

### Architecture

```
┌──────────────────────────┐        SSE         ┌──────────────────────────┐
│  LangChain ReAct agent   │  ───────────────▶  │  FastMCP server          │
│  (this notebook)         │   tool calls /     │  (stock_mcp.py)          │
│                          │   results          │                          │
│  ChatGroq LLM            │  ◀───────────────  │  get_stock_price() →     │
│  + MultiServerMCPClient  │                    │     yfinance             │
└──────────────────────────┘                    └──────────────────────────┘
```

## Install Dependencies

We need four packages:

- **`fastmcp`** — only needed if you also run the server in this environment. The server itself uses it.
- **`yfinance`** — fetches real stock-price history (used by the server, but installed here so the same env can run both).
- **`langchain-mcp-adapters`** — the bridge that turns MCP tools into LangChain `StructuredTool` objects.
- **`langchain-groq` + `langgraph`** — the LLM provider and the underlying graph engine that `create_agent` uses to run the tool-calling loop.

In [11]:
!pip install -q fastmcp yfinance langchain-mcp-adapters langchain-groq langgraph

## Setup LangChain Agent with MultiServerMCPClient

Connect to `stock_price_server.py` via stdio transport using `MultiServerMCPClient`, load the MCP tools, and create a ReAct agent.

The four imports below form the full client toolchain:

- **`MultiServerMCPClient`** — can fan out to *multiple* MCP servers at once. We point it at a single server here, but the same client could aggregate tools from a stocks server, a news server, a calendar server, etc.
- **`create_agent`** — LangChain's built-in ReAct agent factory. Given an LLM and a list of tools, it wires up the **Request → Decide → Execute → Feed Back** loop for you (the same loop you saw built manually in `Tool_Calling.ipynb`).
- **`ChatGroq`** — chat-model wrapper for Groq's hosted Llama / Mixtral models.
- **`asyncio`, `getpass`, `sys`** — standard utilities; the MCP client is async, so we'll `await` calls in cells below.

In [12]:
import os
import sys
import asyncio
from getpass import getpass
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_groq import ChatGroq

`getpass` reads input without echoing it to the screen, so the keys you paste won't end up in cell outputs or notebook checkpoints. Use it whenever you'd otherwise be tempted to hard-code a secret in source.

In [13]:
api_key = getpass("Enter your GROQ_API_KEY: ")

Enter your GROQ_API_KEY: ··········


## Connect to the MCP Server

Two things happen here:

1. **`ChatGroq(...)`** — instantiate the LLM. `llama-3.3-70b-versatile` is Groq's flagship general-purpose model and supports tool calling out of the box.
2. **`MultiServerMCPClient({...})`** — declare the MCP servers we want to talk to. The dict key (`"stock_price"`) is just a label; the value tells the client *how* to reach the server:
   - `transport: "sse"` — Server-Sent Events, an HTTP-based push protocol that works through firewalls and proxies. Matches the transport `stock_mcp.py` chose in its `mcp_server.run(...)` call.
   - `url: "http://127.0.0.1:8100/sse"` — where the server is listening. **The server must already be running** when you execute this cell.

No network call happens yet — `MultiServerMCPClient` is lazy. The actual handshake happens in the next cell when we ask for tools.

In [14]:
import asyncio
import sys
from getpass import getpass
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=api_key)

client = MultiServerMCPClient(
    {
        "stock_price": {
            "transport": "sse",
            "url": "http://127.0.0.1:8100/sse",
        }
    }
)

**`await client.get_tools()`** does the actual MCP handshake: it opens an SSE connection to the server, asks *"what tools do you expose?"*, and converts the schemas the server returns into LangChain `StructuredTool` objects. The list is exactly what the server's `@mcp_server.tool()` decorators registered — in our case, just `get_stock_price`.

**`create_agent(llm, tools)`** builds a ReAct agent that knows how to call those tools. Internally it constructs a LangGraph state machine that runs the four-step tool-calling loop until the model produces a final text answer.

In [15]:
tools = await client.get_tools()
agent = create_agent(llm, tools)

ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)

Inspect what the agent actually sees. The output below shows a single `StructuredTool` with:

- **`name`** — `'get_stock_price'`, taken from the Python function name on the server.
- **`description`** — the function's docstring; this is what the LLM reads to decide *whether* and *how* to call the tool.
- **`args_schema`** — JSON Schema generated from the function signature (`stock_ticker: str`, `duration: str = "1mo"`). The required-vs-optional split comes straight from the default values.
- **`coroutine`** — the actual callable; when invoked it routes back through the SSE connection to the server.

This is the magic of MCP: the *same* tool can be consumed by Claude Desktop, Cursor, an ADK agent, or this LangChain agent — none of them needed to import the server's Python code.

In [ ]:
print(tools)

## Run a Query

Now hand the agent a natural-language question. It will:

1. See the user message + the `get_stock_price` tool schema.
2. Decide to call `get_stock_price(stock_ticker="AAPL", duration="3mo")`.
3. The framework routes that call over SSE to `stock_mcp.py`, which calls `yfinance` and returns a string like *"AAPL gained -7.35% over 3mo."*
4. The LLM reads the tool result and writes a final user-facing answer.

All four steps happen inside the single `await agent.ainvoke(...)` call below. Open the run in LangSmith afterwards to see the full trace.

In [ ]:
response = await agent.ainvoke(
    {"messages": [{"role": "user",
                   "content": "What is the return from AAPL stock in the last 3 months"}]}
)
print(response["messages"][-1].content)

### Multi-step query

The next question requires **two** tool calls — one for AAPL, one for NVDA — followed by a comparison the LLM does in plain text. The agent loop happily makes both calls in sequence; you don't have to write any orchestration code.

If you watch the LangSmith trace, you'll see the assistant emit two tool calls (sometimes in parallel, sometimes sequential, depending on the model's choice) before composing the final answer.

In [ ]:
response = await agent.ainvoke(
    {"messages": [{"role": "user",
                   "content": "compare the return between AAPL and NVIDIA in last 3 months"}]}
)
print(response["messages"][-1].content)